# Geração de Figuras — Relatório IC

**Autor:** Thiago Solé Gomes Heleno

Notebook standalone que produz todas as figuras (PNGs) para o relatório
`relatorio_ic.tex`. Não modifica notebooks de pipeline existentes.

**Saídas:** pasta `figs/` na raiz do repositório.

**Seções:**
1. Configuração
2. EDA — Wind Farm C (desbalanceamento, séries, correlação, faltantes)
3. Métricas consolidadas — comparação dos 5 NBs
4. Diagramas auxiliares


## 1. Configuração

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# Caminhos
ROOT = Path('..').resolve() if Path('notebooks').exists() else Path('.').resolve()
if (ROOT / 'CARE_To_Compare').exists():
    pass
else:
    # fallback: assume notebook ran from repo root
    ROOT = Path('.').resolve()

WF_C = ROOT / 'CARE_To_Compare' / 'Wind Farm C'
RESULTADOS = ROOT / 'resultados'
FIGS = ROOT / 'figs'
FIGS.mkdir(exist_ok=True)

print(f'ROOT     : {ROOT}')
print(f'WF_C     : {WF_C}')
print(f'FIGS     : {FIGS}')

# Estilo padrão
plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 200,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})
sns.set_palette('deep')

## 2. EDA — Wind Farm C

### 2.1 Desbalanceamento de classes (eventos e amostras)

In [ ]:
events = pd.read_csv(WF_C / 'event_info.csv', sep=';')
print(events['event_label'].value_counts())
events.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Painel esquerdo: contagem de eventos
label_counts = events['event_label'].value_counts()
axes[0].bar(label_counts.index, label_counts.values,
            color=['#2E86AB', '#C73E1D'], edgecolor='black')
axes[0].set_title('Distribuição de Eventos — Wind Farm C')
axes[0].set_ylabel('Número de eventos')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 0.5, str(v), ha='center', fontweight='bold')

# Painel direito: proporção amostral 98:2 (do analise_comparativa.md)
samples_normal = 3_126_239
samples_anomaly = 60_897
axes[1].bar(['Normal', 'Anomalia'], [samples_normal, samples_anomaly],
            color=['#2E86AB', '#C73E1D'], edgecolor='black')
axes[1].set_title('Distribuição de Amostras (10-min) — desbalanceamento 51:1')
axes[1].set_ylabel('Número de amostras')
axes[1].set_yscale('log')
for i, v in enumerate([samples_normal, samples_anomaly]):
    axes[1].text(i, v * 1.2, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(FIGS / 'eda_desbalanceamento.png')
plt.show()

### 2.2 Série temporal — exemplo de evento anômalo

In [ ]:
# Pega primeiro evento anômalo
first_anom = events[events['event_label'] == 'anomaly'].iloc[0]
asset_id = int(first_anom['asset_id'])
event_id = int(first_anom['event_id'])
event_start = pd.to_datetime(first_anom['event_start'])
event_end = pd.to_datetime(first_anom['event_end'])

print(f'Evento {event_id}, asset {asset_id}: {event_start} -> {event_end}')

csv_path = WF_C / 'datasets' / f'{event_id}.csv'
df = pd.read_csv(csv_path, sep=';', parse_dates=['time_stamp'])
df.head()

In [ ]:
# Identifica colunas de potência e velocidade do vento (heurística)
power_cols = [c for c in df.columns if 'power' in c.lower() or 'sensor_0_avg' in c.lower()]
wind_cols  = [c for c in df.columns if 'wind' in c.lower() or 'sensor_1_avg' in c.lower()]
print('Power cols (top 3):', power_cols[:3])
print('Wind cols  (top 3):', wind_cols[:3])

# Plota se encontrou
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
if power_cols:
    axes[0].plot(df['time_stamp'], df[power_cols[0]], color='#2E86AB', linewidth=0.5)
    axes[0].axvspan(event_start, event_end, color='red', alpha=0.15, label='Período anômalo')
    axes[0].set_ylabel(power_cols[0])
    axes[0].set_title(f'Série temporal — Evento {event_id} (asset {asset_id})')
    axes[0].legend(loc='upper right')
if wind_cols:
    axes[1].plot(df['time_stamp'], df[wind_cols[0]], color='#06A77D', linewidth=0.5)
    axes[1].axvspan(event_start, event_end, color='red', alpha=0.15)
    axes[1].set_ylabel(wind_cols[0])
    axes[1].set_xlabel('Timestamp')

plt.tight_layout()
plt.savefig(FIGS / 'eda_serie_temporal_evento.png')
plt.show()

### 2.3 Mapa de calor — valores faltantes

In [ ]:
# Amostragem para evitar mapa enorme
sample = df.iloc[::50, :]   # 1 a cada 50 linhas
numeric = sample.select_dtypes(include=[np.number])
# Top 30 colunas com mais NaN
nan_cols = numeric.isna().sum().sort_values(ascending=False).head(30).index.tolist()

if nan_cols:
    fig, ax = plt.subplots(figsize=(12, 7))
    sns.heatmap(numeric[nan_cols].isna().T, cbar=False, cmap='RdYlGn_r',
                xticklabels=False, yticklabels=True, ax=ax)
    ax.set_title(f'Padrão de Valores Faltantes — Top 30 colunas (Evento {event_id})')
    ax.set_xlabel('Tempo (amostrado)')
    plt.tight_layout()
    plt.savefig(FIGS / 'eda_faltantes_heatmap.png')
    plt.show()
else:
    print('Sem NaN visíveis no sample')

### 2.4 Matriz de correlação — subset de features

In [ ]:
# Carrega lista de features selecionadas (NB2)
feat_path = RESULTADOS / '02_cnn_bilstm_autoencoder' / 'features_selecionadas.json'
if feat_path.exists():
    with open(feat_path) as f:
        sel = json.load(f)
    if isinstance(sel, dict):
        feats = sel.get('features_selecionadas', sel.get('features', []))[:20]
    else:
        feats = sel[:20]
else:
    feats = numeric.columns[:20].tolist()

# Filtra só features existentes no df
feats = [f for f in feats if f in df.columns][:20]
print(f'Usando {len(feats)} features para correlação')

if feats:
    corr = df[feats].corr()
    fig, ax = plt.subplots(figsize=(10, 9))
    sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                square=True, annot=False, ax=ax,
                cbar_kws={'label': 'Coeficiente de Correlação'})
    ax.set_title(f'Matriz de Correlação — {len(feats)} features SCADA')
    plt.tight_layout()
    plt.savefig(FIGS / 'eda_matriz_correlacao.png')
    plt.show()

### 2.5 Distribuição de variáveis SCADA — histogramas

In [ ]:
if feats:
    n_show = min(9, len(feats))
    cols = 3
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(14, rows * 3))
    axes = axes.flatten() if rows > 1 else [axes] if n_show == 1 else axes
    for i in range(n_show):
        ax = axes[i]
        data = df[feats[i]].dropna()
        ax.hist(data, bins=50, color='#2E86AB', edgecolor='black', alpha=0.7)
        ax.set_title(feats[i][:35], fontsize=9)
        ax.set_ylabel('Freq.')
    for j in range(n_show, len(axes)):
        axes[j].axis('off')
    plt.suptitle('Distribuição de Variáveis SCADA Selecionadas', fontsize=13, y=1.00)
    plt.tight_layout()
    plt.savefig(FIGS / 'eda_distribuicoes_features.png')
    plt.show()

## 3. Métricas Consolidadas — Comparação dos 5 NBs

In [ ]:
metrics = pd.read_csv(RESULTADOS / 'metricas_consolidadas.csv')
print(metrics.shape)
metrics.head(10)

### 3.1 Comparação F1-Score por configuração

In [ ]:
# Filtra linhas com F1 não nulo
df_f1 = metrics[metrics['f1'].notna() & (metrics['f1'] > 0)].copy()
df_f1['label'] = df_f1['notebook'].str.replace('wind_turbine_', '', regex=False) + '\n' + \
                 df_f1['modelo'].fillna('') + '\n(' + df_f1['configuracao'].fillna('') + ')'
df_f1 = df_f1.sort_values('f1', ascending=True)

fig, ax = plt.subplots(figsize=(12, max(6, len(df_f1) * 0.35)))
colors = ['#C73E1D' if 'supervisionado' == p and 'semi' not in p else '#2E86AB'
          for p in df_f1['paradigma']]
ax.barh(df_f1['label'], df_f1['f1'], color=colors, edgecolor='black')
ax.set_xlabel('F1-Score (amostra/janela)')
ax.set_title('Comparação F1-Score — todas as configurações executadas')
for i, v in enumerate(df_f1['f1']):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / 'comparacao_f1_score.png')
plt.show()

### 3.2 Comparação AUC-ROC

In [ ]:
df_auc = metrics[metrics['auc_roc'].notna() & (metrics['auc_roc'] > 0)].copy()
df_auc['label'] = df_auc['modelo'].fillna('') + '\n(' + df_auc['configuracao'].fillna('') + ')'
df_auc = df_auc.sort_values('auc_roc', ascending=True)

fig, ax = plt.subplots(figsize=(11, max(5, len(df_auc) * 0.4)))
ax.barh(df_auc['label'], df_auc['auc_roc'], color='#06A77D', edgecolor='black')
ax.axvline(0.5, color='gray', linestyle='--', label='Aleatório (0.5)')
ax.set_xlabel('AUC-ROC')
ax.set_xlim(0, 1)
ax.set_title('Comparação AUC-ROC — modelos com avaliação amostral')
ax.legend()
for i, v in enumerate(df_auc['auc_roc']):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / 'comparacao_auc_roc.png')
plt.show()

### 3.3 Comparação CARE Score

In [ ]:
df_care = metrics[metrics['CARE_score'].notna() & (metrics['CARE_score'] > 0.001)].copy()
df_care['label'] = df_care['notebook'].str.replace('wind_turbine_', '', regex=False) + '\n(' + \
                   df_care['configuracao'].fillna('') + ')'
df_care = df_care.sort_values('CARE_score', ascending=True)

fig, ax = plt.subplots(figsize=(11, max(4, len(df_care) * 0.5)))
ax.barh(df_care['label'], df_care['CARE_score'], color='#A23B72', edgecolor='black')
ax.set_xlabel('CARE Score')
ax.set_xlim(0, 1)
ax.set_title('Comparação CARE Score (framework adotado de EnergyFaultDetector)')
for i, v in enumerate(df_care['CARE_score']):
    ax.text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / 'comparacao_care_score.png')
plt.show()

### 3.4 Precision vs Recall — visão por amostra

In [ ]:
df_pr = metrics[metrics['precision'].notna() & metrics['recall'].notna() &
                (metrics['split'] == 'teste')].copy()
df_pr['label'] = df_pr['modelo'].fillna('') + ' — ' + df_pr['configuracao'].fillna('')

fig, ax = plt.subplots(figsize=(11, 7))
scatter = ax.scatter(df_pr['recall'], df_pr['precision'],
                     s=120, c=range(len(df_pr)), cmap='tab20',
                     edgecolor='black', linewidth=0.8)
for _, row in df_pr.iterrows():
    ax.annotate(row['label'][:30], (row['recall'], row['precision']),
                fontsize=7, xytext=(5, 5), textcoords='offset points')
ax.set_xlabel('Recall (sensibilidade)')
ax.set_ylabel('Precision (precisão)')
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.set_title('Precision vs Recall — todos os modelos no conjunto de teste')
plt.tight_layout()
plt.savefig(FIGS / 'comparacao_precision_recall.png')
plt.show()

## 4. Diagramas de Pipeline

Para diagramas de arquitetura (LSTM gates, CNN1D conv, MultiHeadAttention block,
pipeline geral) recomenda-se gerar via TikZ no LaTeX, draw.io, ou Netron a
partir dos modelos `.keras` salvos em `resultados/03/`, `04/` e `05/`.

Estes diagramas não podem ser produzidos automaticamente sem ferramentas
externas — placeholders mantidos no `relatorio_ic.tex`.

## 5. Lista de Figuras Geradas

In [ ]:
geradas = sorted(FIGS.glob('*.png'))
print(f'Total: {len(geradas)} PNGs em {FIGS}')
for p in geradas:
    print(f'  {p.name}  ({p.stat().st_size / 1024:.1f} KB)')